## tl;dr

The prior 65-to-100 universe transition coincided with a lower degenerate-retrain rate (65.6% to 53.1%), but it did not solve the problem and was bundled with USD accounting and listing-mask changes. Further investable-universe expansion is therefore only suggestive, not a clean next causal arm. A model-only shadow universe is the safer structural experiment.

## Context & Methods

Decision: whether expanding the current 100-name universe is more likely than `val_window=252` to reduce `n_trees < 10` retrain degeneracy.

### Key Assumptions

- The pre-S9 artifact at commit `528e752` is the 65-name baseline described in the decision log.
- The current artifact is the 100-name, USD-accounted baseline.
- A simple `1/sqrt(N)` calculation is an optimistic independence benchmark, not a forecast, because stock returns and relevance labels are correlated.

## Data

Sources: current `outputs/codex_causal_rank_65/metrics.json`, the same path at commit `528e752`, `src/model_trainer.py`, and the S9/S10 decision log.

In [ ]:
from pathlib import Path
import json
import math
import subprocess

root = Path.cwd()
current_path = root / 'outputs' / 'codex_causal_rank_65' / 'metrics.json'
current = json.loads(current_path.read_text(encoding='utf-8'))
old_text = subprocess.check_output([
    'git', '-c', f'safe.directory={root.as_posix()}', 'show',
    '528e752:outputs/codex_causal_rank_65/metrics.json'
], cwd=root, text=True, encoding='utf-8')
old = json.loads(old_text)


In [ ]:
old_events = {row['date'] for row in old['model_quality']['events']}
new_events = {row['date'] for row in current['model_quality']['events']}
comparison = {
    'old_universe': 65,
    'current_universe': current['data_quality']['universe']['loaded_ticker_count'],
    'old_degenerate': old['model_quality']['degenerate_retrains'],
    'current_degenerate': current['model_quality']['degenerate_retrains'],
    'old_rate': old['model_quality']['degenerate_rate'],
    'current_rate': current['model_quality']['degenerate_rate'],
    'rescued_events': len(old_events - new_events),
    'newly_degenerate_events': len(new_events - old_events),
    'common_degenerate_events': len(old_events & new_events),
    'ir_old': old['metrics']['information_ratio'],
    'ir_current': current['metrics']['information_ratio'],
}
comparison


In [ ]:
independence_benchmark = [
    {'move': '65 to 100', 'optimistic_se_reduction': 1 - math.sqrt(65 / 100)},
    {'move': '100 to 130', 'optimistic_se_reduction': 1 - math.sqrt(100 / 130)},
    {'move': '100 to 150', 'optimistic_se_reduction': 1 - math.sqrt(100 / 150)},
]
independence_benchmark


## Results

- Degenerate retrains fell from 21/32 to 17/32, a 12.5 percentage-point improvement.
- Event transitions were 11 rescued, 7 newly degenerate, and 10 degenerate under both baselines. The direction is favorable but the discordant sample is small.
- The production IR moved from about 1.697 to 1.570, but this is not an apples-to-apples performance comparison because universe, currency accounting, listing masks, and later risk-cap settings changed.
- The current ranker already has 100 loaded names, with four recent-listing history gates. Further expansion has diminishing statistical benefit and can change relevance-label composition, sector mix, benchmark coverage, missing-data imputation, and survivorship exposure.

## Takeaways

1. Universe size is probably a contributing driver, but not the dominant remaining cause of degeneracy.
2. Keep `val_window=252` as the cleaner next one-parameter arm.
3. If universe expansion is tested, use a 130-150-name model-only shadow universe while keeping the investable and portfolio-evaluation universe fixed at 100. Pre-register point-in-time eligibility, sector/country coverage, validation NDCG, best iteration, degenerate transitions, IR/TE/beta/turnover guards, and recent-period IR.